# OpenAI API 입문과 한국어 음성 AI 실습

**음성 모델 본과정에 들어가기 전, 먼저 완성된 AI를 사용해 보는 사전 실습 교재**

> **최종 결과물:** 한국어 질문 음성(WAV) → Whisper 음성 인식(STT) → LLM 답변 생성 → 음성 합성(TTS) → 결과 재생

이 교재만 보고 실습할 수 있도록 **왜 배우는지 → 무엇을 할지 → 실행 방법 → 정상 결과 → 자율 실습** 순서로 구성했습니다.

### 학습 흐름

| 단계 | 주제 | 실습 결과 |
|---|---|---|
| API 기초 | API·JSON·인증 | 첫 LLM 호출 |
| 텍스트 AI | 질문·대화 기록·오류 처리 | 한국어 대화 도우미 |
| 음성 생성 | OpenAI TTS | 한국어 WAV 파일 |
| 음성 인식 | Whisper STT 및 인식 결과 확인 | 한국어 받아쓰기 |
| 기능 통합 | STT → LLM → TTS | 한국어 음성 질문·답변 |
| 자율 프로젝트 | 입력 변경·결과 검증·기록 | 개인별 음성 AI 실습 결과 |


## 0. 시작 전 준비와 안전 수칙

1. 인터넷이 연결된 Jupyter 환경에서 진행합니다. Python 3.10 이상 권장.
2. 제공된 `.env` 파일을 노트북과 같은 폴더에 배치합니다. **ChatGPT 구독과 API 사용 요금은 별개**입니다.
3. 수업용 음성에는 본인의 음성 또는 공유 동의가 있는 파일만 사용합니다. 타인의 음성을 허락 없이 업로드하지 않습니다.
4. `.env`와 API 키를 제출·공유하지 않습니다. 키를 코드와 GitHub에 직접 넣지 않습니다.
5. 실제 호출 실패 시에는 **오류 메시지를 읽고** 인증, 잔액/한도, 모델 접근 권한, 네트워크를 차례로 확인합니다.

이번 과정은 API 활용 수업이지 Whisper 내부 구조 수업이 아닙니다. **다음 40시간 음성 모델 수업에서 파형, 샘플링, FFT/STFT, Mel, 모델 구조, 평가를 자세히 배웁니다.**

In [1]:
# 필요한 패키지를 설치합니다. (최초 1회)
%pip install -q -U openai python-dotenv numpy soundfile ipython

Note: you may need to restart the kernel to use updated packages.


  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.3.12 requires openai<2.0.0,>=1.68.2, but you have openai 2.48.0 which is incompatible.


### 실습 폴더 준비
- `voice_api_lab/`: 수업 중 생성한 파일을 보관합니다.
- `voice_api_lab/input/`: 입력 음성 파일
- `voice_api_lab/output/`: 합성 음성·중간 결과
- `voice_api_lab/report/`: 검증 기록

다른 폴더에서 노트북을 실행해도 아래 상대 경로를 기준으로 생성됩니다.

In [2]:
from pathlib import Path

# 입력 음성·출력 음성·실습 기록을 저장할 폴더를 준비합니다.
BASE = Path('voice_api_lab')
INPUT_DIR = BASE / 'input'
OUTPUT_DIR = BASE / 'output'
REPORT_DIR = BASE / 'report'
for folder in (INPUT_DIR, OUTPUT_DIR, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print('실습 폴더:', BASE.resolve())

실습 폴더: G:\OneDrive\AISource\2026_구름\음성\voice_api_lab


# Part 1. API 기초와 LLM 활용
## 왜 API를 먼저 배울까요?

**목적:** 직접 모델을 훈련하지 않고도 외부 AI 서비스를 파이썬 코드에서 사용할 수 있음을 경험합니다.

- **AI 모델**: 입력을 받아 추론하고 출력을 만드는 부분
- **API**: 우리가 만든 프로그램이 다른 서비스에 요청하는 약속/접점
- **SDK**: HTTP 요청 작성 등의 번거로움을 줄여 주는 라이브러리
- **JSON**: API에서 데이터를 주고받을 때 널리 사용하는 텍스트 형식

요청(질문, 설정) → API 서버 → 모델 처리 → 응답(결과) 순서입니다. API를 호출했다고 모델을 직접 학습시키는 것은 아닙니다.

### 실습 A: JSON ↔ 딕셔너리

**왜 하나요?** API 응답에서 필요한 항목을 추출하려면 구조를 읽을 수 있어야 합니다.

In [3]:
import json

# 모델에 전달할 요청 내용을 파이썬 딕셔너리로 표현합니다.
request_dict = {
    'model': 'gpt-4.1-mini',
    'input': '한국어로 음성 인식의 뜻을 설명해 줘.',
    'max_output_tokens': 120,
}
# 딕셔너리를 JSON 문자열로 바꾸고 다시 복원합니다.
request_json = json.dumps(request_dict, ensure_ascii=False, indent=2)
print('JSON 문자열:\n', request_json)
restored = json.loads(request_json)
print('복원된 딕셔너리의 모델:', restored['model'])
print('타입:', type(request_json).__name__, '→', type(restored).__name__)

JSON 문자열:
 {
  "model": "gpt-4.1-mini",
  "input": "한국어로 음성 인식의 뜻을 설명해 줘.",
  "max_output_tokens": 120
}
복원된 딕셔너리의 모델: gpt-4.1-mini
타입: str → dict


**정상 결과:** JSON 문자열이 표시되고 `str → dict`가 출력됩니다. 여기까지는 API 요금이 발생하지 않습니다.

**직접 해보기**: `input`을 자신의 질문으로 바꾸고, 딕셔너리에 `purpose` 키를 추가한 뒤 다시 JSON으로 변환하세요.

**점검 질문:** JSON과 파이썬 딕셔너리는 왜 구분해야 할까요? → JSON은 전송 가능한 텍스트, 딕셔너리는 파이썬에서 다루는 자료구조입니다.

## API 인증과 첫 호출 준비

수업에서 제공된 `.env` 파일을 **노트북과 같은 폴더**에 두세요. 파일에는 `OPENAI_API_KEY=...` 형식의 설정이 들어 있습니다.

- API 키를 코드에 직접 적거나 제출물에 포함하지 않습니다.
- 모든 실습은 실제 OpenAI API를 호출합니다. 인터넷 연결과 API 사용 권한이 필요합니다.
- 음성 파일은 API 서버로 전송되므로 공개·사용 허가된 파일 또는 자신의 음성만 사용합니다.
- 반복 실행할 때마다 사용량에 따른 요금이 발생할 수 있습니다.

In [4]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# 수업에서 제공된 .env 파일을 읽습니다.
load_dotenv(override=True)

# 각 실습에서 사용할 AI 모델을 설정합니다.
TEXT_MODEL = 'gpt-4.1-mini'
STT_MODEL = 'whisper-1'
TTS_MODEL = 'gpt-4o-mini-tts'

# API 키가 있는지 확인하고 실제 API 클라이언트를 생성합니다.
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY를 찾지 못했습니다. 노트북 실행 폴더의 .env 파일을 확인하세요.')
client = OpenAI()
print('API 키 확인 완료: 실제 API 요청을 사용합니다.')

API 키 확인 완료: 실제 API 요청을 사용합니다.


### 실습 B: 텍스트 생성

**목적:** 모델 호출에서 `model`, `input`, `response.output_text`가 각각 무엇인지 확인합니다.

**예상 결과:** '음성 인식'에 대한 한국어 설명이 반환됩니다. 모델의 응답은 매번 조금 달라질 수 있습니다.

In [5]:
# 질문을 실제 LLM API에 전달하고 답변을 출력합니다.
question = '음성 인식(STT)이 무엇인지 파이썬 초보자에게 두 문장으로 설명해 주세요.'
response = client.responses.create(
    model=TEXT_MODEL,
    input=question,
    max_output_tokens=180,
)
answer = response.output_text  # 모델이 생성한 텍스트만 추출합니다.
print('질문:', question)
print('답변:', answer)

질문: 음성 인식(STT)이 무엇인지 파이썬 초보자에게 두 문장으로 설명해 주세요.
답변: 음성 인식(STT, Speech-to-Text)은 사람들이 말한 소리를 컴퓨터가 텍스트로 바꾸는 기술입니다. 파이썬에서는 라이브러리를 사용해 간단히 음성을 글자로 변환할 수 있어요.


**직접 해보기:** 질문을 세 가지로 바꾸어 보세요. (1) 한 문장만 답변 (2) 초등학생 눈높이 (3) 음성 인식 오류 원인 세 가지. '질문의 변화가 결과를 어떻게 바꾸는가'를 두 줄로 기록합니다.



## 지시문과 대화 이력

### 왜 필요한가요?
사용자가 음성으로 '그 내용을 더 쉽게 설명해 줘'라고 말하면, AI가 **앞 질문을 기억하도록 프로그램이 이전 대화 내용을 함께 전달**해야 할 수 있습니다. 모델 호출 자체와 대화 기록 관리는 구분해야 합니다.

실습에서는 메시지 배열을 직접 관리합니다. 한 번에 보내는 데이터가 많아지면 비용·컨텍스트 길이가 늘어날 수 있습니다.

In [6]:
# 여러 질문에 이전 대화가 이어지도록 기록을 준비합니다.
history = [
    {'role': 'developer', 'content': '너는 한국어 AI 교육 도우미이다. 쉬운 말로 두 문장 이내로 답한다.'}
]

def ask_with_history(user_text):
    # 새 질문과 과거 메시지를 함께 모델에 전달합니다.
    history.append({'role': 'user', 'content': user_text})
    result = client.responses.create(
        model=TEXT_MODEL,
        input=history,
        max_output_tokens=180,
    )
    reply = result.output_text
    history.append({'role': 'assistant', 'content': reply})  # 답변도 다음 대화를 위해 저장합니다.
    return reply

print(ask_with_history('Whisper는 어떤 기능을 하나요?'))
print(ask_with_history('그 기능을 일상생활 예시로 설명해 주세요.'))
print('대화 항목 수:', len(history))

Whisper는 사람의 말을 텍스트로 바꿔주는 음성 인식 기능이에요. 녹음된 음성을 문자로 쉽게 변환할 수 있어요.
Whisper를 사용하면 친구와 통화 내용을 자동으로 문자로 받아볼 수 있어요. 그래서 중요한 말을 놓치지 않고 기록할 수 있어요.
대화 항목 수: 5


**직접 해보기:** `history`에 저장된 메시지의 역할(`developer`, `user`, `assistant`)을 확인하고, '앞 답변을 한 줄로 요약해 줘'를 추가합니다.

**생각해 보기:** 같은 질문을 이전 이력 없이 다시 보내면 무엇이 달라질까요?

## 안전한 요청 함수 만들기

실무에서는 키가 없거나 인터넷이 끊겨도 프로그램이 갑자기 중단되지 않게 처리해야 합니다. 아래 함수는 학습용 최소 예시입니다. 실제 배포 시에는 로그, 재시도 정책, 사용량 제한 등을 보완해야 합니다.

In [7]:
def generate_text(prompt, max_tokens=180):
    """한국어 질문을 실제 LLM에 보내고 성공 여부와 답변을 반환합니다."""
    # 빈 질문은 API에 전송하지 않습니다.
    if not isinstance(prompt, str) or not prompt.strip():
        return False, '질문이 비어 있습니다.'
    try:
        # LLM에 질문을 보내고 텍스트 응답을 받습니다.
        res = client.responses.create(
            model=TEXT_MODEL,
            input=prompt,
            max_output_tokens=max_tokens,
        )
        return True, res.output_text
    except Exception as exc:
        # 민감한 오류 원문 대신 오류 종류만 출력합니다.
        return False, f'API 요청 실패 ({type(exc).__name__}): 키·결제·네트워크·모델을 확인하세요.'

# 잘못된 입력과 정상 질문을 각각 점검합니다.
for sample in ['', '텍스트 음성 합성을 한 문장으로 설명해 줘.']:
    success, result = generate_text(sample)
    print('성공:', success, '| 결과:', result)

성공: False | 결과: 질문이 비어 있습니다.
성공: True | 결과: 텍스트 음성 합성은 입력된 글자를 자연스러운 음성으로 변환하는 기술입니다.


**API 기초 실습 정리 / 제출**

1. API와 AI 모델의 차이를 세 문장 이내로 설명합니다.
2. JSON 요청/응답에서 `model`과 `input`의 의미를 적습니다.
3. 두 개 이상 이어지는 질문으로 대화 기록 사용 여부를 실험합니다.
4. 호출 실패 상황(키 없음, 빈 질문, 한도 초과 등) 중 한 가지에 대한 처리 방법을 기록합니다.

**학습 점검:** OpenAI API를 사용해도 Whisper 내부 신호 처리 원리를 이해한 것은 아닙니다. 다음 단계에서는 먼저 기능을 경험하고, 이후 음성 모델 수업에서 원리를 살펴봅니다.

# Part 2. 한국어 음성 AI 실습
## TTS: 텍스트를 한국어 음성으로 만들기

### 먼저 TTS를 하는 이유
STT를 실습하려면 실제 음성 파일이 필요합니다. 학생마다 마이크 설정이 달라 실습이 중단될 수 있으므로, **한국어 합성 음성을 만들어 공통 입력 예시로 사용**합니다. 이후 본인이 녹음한 음성으로 교체합니다.

- **입력:** 한국어 문장
- **출력:** WAV 오디오 파일
- **핵심 질문:** 모델이 텍스트만 받아 어떻게 소리를 만들까? (원리는 이후 수업에서 학습)

> 합성된 음성은 실제 사람의 자발적 발화와 특성이 다릅니다. 합성 음성으로 STT가 잘 되었다고 실제 환경에서도 같은 인식률을 보장하지 않습니다.

In [8]:
from IPython.display import Audio, display

# 이후 Whisper에 입력할 한국어 질문 음성을 만듭니다.
TTS_SAMPLE_TEXT = '음성 인식과 음성 합성은 어떻게 다른가요?'
TTS_SAMPLE = INPUT_DIR / 'korean_tts_sample.wav'

def text_to_wav(text, path):
    """실제 TTS API로 한국어 음성을 만들어 WAV로 저장합니다."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError('합성할 문장을 입력하세요.')
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)  # 저장 폴더가 없으면 생성합니다.

    # 음성 데이터가 도착하는 대로 WAV 파일에 기록합니다.
    with client.audio.speech.with_streaming_response.create(
        model=TTS_MODEL,
        voice='alloy',
        input=text,
        response_format='wav',
    ) as response:
        response.stream_to_file(path)
    return path

sample_path = text_to_wav(TTS_SAMPLE_TEXT, TTS_SAMPLE)
print('음성 파일 생성:', sample_path)
display(Audio(filename=str(sample_path)))  # 생성된 한국어 음성을 재생합니다.

음성 파일 생성: voice_api_lab\input\korean_tts_sample.wav


**정상 결과:** `voice_api_lab/input/korean_tts_sample.wav` 파일이 생성되고 한국어 질문 음성이 재생됩니다.

**직접 해보기:** 한국어 문장 두 개를 각각 합성한 뒤, 음성 길이와 발음의 자연스러움을 비교하세요. 각 API 호출에는 사용량에 따른 요금이 발생할 수 있습니다.

## Whisper: 한국어 음성 파일을 텍스트로 인식하기

### 실습 목적
**Whisper API**에 WAV 파일을 올려서 음성을 텍스트로 변환합니다. Whisper는 음성 인식 기능을 제공하지만, 이 API를 사용한다고 모델을 로컬 PC에 다운로드하거나 직접 학습시키는 것은 아닙니다.

**준비할 파일:** 방금 만든 `korean_tts_sample.wav` 또는 직접 녹음한 한국어 WAV 파일.

직접 녹음 파일을 쓴다면 아래 `STT_INPUT`만 바꾸세요. 예: `Path('my_voice.wav')`.

In [9]:
import wave

# TTS로 만든 음성 또는 직접 준비한 WAV를 입력으로 선택합니다.
STT_INPUT = TTS_SAMPLE

def inspect_wav(path):
    """WAV 파일의 채널·샘플링 주파수·길이를 확인합니다."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'음성 파일이 없습니다: {path}')
    try:
        # WAV 헤더에서 음성의 기본 속성을 읽습니다.
        with wave.open(str(path), 'rb') as wf:
            info = {
                'channels': wf.getnchannels(),
                'sample_rate': wf.getframerate(),
                'frames': wf.getnframes(),
                'duration_sec': round(wf.getnframes() / wf.getframerate(), 2),
            }
    except wave.Error as exc:
        raise ValueError('표준 PCM WAV 헤더를 읽지 못했습니다. 파일 형식을 확인하세요.') from exc
    print(info)
    return info

wav_info = inspect_wav(STT_INPUT)

{'channels': 1, 'sample_rate': 24000, 'frames': 2147483647, 'duration_sec': 89478.49}


### 실습 C: Whisper로 받아쓰기

입력 음성을 한국어(`language='ko'`)로 인식하도록 요청합니다. **실제 호출 시 파일 내용이 외부 API로 전송됩니다.**

In [10]:
def transcribe_korean(path, model=STT_MODEL):
    """Whisper API로 한국어 음성을 인식하고 텍스트를 반환합니다."""
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'음성 파일이 없습니다: {path}')

    # 음성 파일을 열어 실제 Whisper API에 전송합니다.
    with path.open('rb') as f:
        result = client.audio.transcriptions.create(
            model=model,
            file=f,
            language='ko',  # 한국어 음성임을 지정합니다.
        )
    return result.text

# 인식 결과를 이후 비교 실습과 통합 실습에서 사용합니다.
recognized_text = transcribe_korean(STT_INPUT)
print('인식 결과:', recognized_text)

인식 결과: 음성인식과 음성합성은 어떻게 다른가요?


### 실습 D: 정답 문장과 인식 결과를 비교하기

정답을 아는 TTS 파일일 때만 `TTS_SAMPLE_TEXT`를 기준 문장으로 활용합니다. 개인 녹음 파일은 **실제로 말한 내용을 직접 정답으로 기록**해야 합니다.

여기서는 학습자가 결과를 읽고 차이를 확인하는 수준으로 진행합니다. **CER/WER 등 정량 평가는 이후 음성 모델 수업에서** 배우게 됩니다.

In [11]:
# TTS 문장의 경우 원래 문장을 정답으로 사용합니다.
reference_text = TTS_SAMPLE_TEXT if STT_INPUT == TTS_SAMPLE else '직접 들은 발화를 작성하세요.'
print('정답 문장:', reference_text)
print('인식 문장:', recognized_text)

# 실제 음성을 듣고 차이점과 원인 가설을 직접 작성합니다.
review = {
    'source_file': str(STT_INPUT),
    'reference_text': reference_text,
    'recognized_text': recognized_text,
    'difference_note': '미작성: 빠진 단어, 바뀐 단어, 문장부호 차이를 기록하세요.',
    'possible_cause': '미작성: 발음, 말속도, 잡음 등의 가능성을 적으세요.',
}
print(json.dumps(review, ensure_ascii=False, indent=2))

정답 문장: 음성 인식과 음성 합성은 어떻게 다른가요?
인식 문장: 음성인식과 음성합성은 어떻게 다른가요?
{
  "source_file": "voice_api_lab\\input\\korean_tts_sample.wav",
  "reference_text": "음성 인식과 음성 합성은 어떻게 다른가요?",
  "recognized_text": "음성인식과 음성합성은 어떻게 다른가요?",
  "difference_note": "미작성: 빠진 단어, 바뀐 단어, 문장부호 차이를 기록하세요.",
  "possible_cause": "미작성: 발음, 말속도, 잡음 등의 가능성을 적으세요."
}


**직접 해보기**

1. 합성 음성과 자신의 녹음 음성 중 한 가지를 Whisper로 인식합니다.
2. 음성을 직접 듣고 '실제로 말한 문장(정답)'을 작성합니다.
3. 인식 결과에서 빠진 단어나 바뀐 단어를 한 가지 이상 찾습니다. 차이가 없다면 '차이 없음'으로 기록합니다.
4. 인식 오류가 발생했다면 어떤 조건이 영향을 주었을지 **가설**을 적습니다. 원인을 단정하지 마세요.

**선택 비교:** `STT_MODEL='gpt-4o-mini-transcribe'`로 바꿔 같은 파일을 다시 인식해 보세요. 모델별 접근 가능 여부와 요금은 계정 설정에 따라 다릅니다. 정확도 비교는 **한 문장만으로 일반화하지 않습니다.**

## 음성 질문 → LLM 답변 → TTS 응답 통합

### 이 실습에서 하는 일

1. **STT:** 한국어 음성 파일을 받아 문장으로 변환
2. **LLM:** 변환한 문장의 의미를 읽고 한국어로 답변 생성
3. **TTS:** 답변 문장을 다시 음성 파일로 변환

이 셀은 여러 API 호출이 발생할 수 있으므로 **한 번의 실행에 최대 STT 1회 + LLM 1회 + TTS 1회**만 수행합니다. 무한 반복은 하지 않습니다.

**주의:** AI가 생성한 답변의 사실성은 보장되지 않습니다. 기밀 자료·개인정보가 들어 있는 음성은 이 실습에 사용하지 않습니다.

In [ ]:
# 음성 입력과 답변 음성 저장 경로를 지정합니다.
FINAL_INPUT = STT_INPUT
FINAL_OUTPUT = OUTPUT_DIR / 'voice_assistant_answer.wav'

def run_voice_assistant(audio_path, save_path):
    """실제 STT → LLM → TTS API를 연결해 음성 질문에 답합니다."""
    audio_path = Path(audio_path)
    if not audio_path.is_file():
        raise FileNotFoundError(f'입력 WAV가 없습니다: {audio_path}')

    # 1. 음성을 한국어 문장으로 변환합니다.
    user_text = transcribe_korean(audio_path)

    # 2. 인식된 질문에 대해 LLM 답변을 생성합니다.
    success, reply = generate_text(
        '다음 음성 질문에 친절하게 한국어 두 문장 이내로 대답해 주세요.\n'
        f'질문: {user_text}'
    )
    if not success:
        raise RuntimeError(f'LLM 답변 생성 실패: {reply}')

    # 3. 답변을 한국어 음성 WAV로 저장합니다.
    output_path = text_to_wav(reply, save_path)
    return {'question': user_text, 'answer': reply, 'audio_file': output_path}

result = run_voice_assistant(FINAL_INPUT, FINAL_OUTPUT)
print('질문:', result['question'])
print('답변:', result['answer'])
display(Audio(filename=str(result['audio_file'])))

**정상 결과:** 음성 질문의 텍스트, LLM 답변 텍스트, 답변 WAV 파일이 생성됩니다. 정상 동작 시 `voice_api_lab/output/voice_assistant_answer.wav`를 확인하세요.

**직접 해보기:** 입력 음성을 '음성 인식과 음성 합성의 차이는 무엇인가요?'로 바꿉니다. 3단계 중 각각 어느 서비스가 무엇을 처리했는지 한 문장씩 작성합니다.

## 자율 실습 프로젝트 — 나만의 한국어 음성 안내 도우미

### 프로젝트 목표
앞서 작성한 코드를 활용해 **한국어 음성 한 개를 입력받아 한국어 음성 답변을 출력**하는 간단한 프로그램을 완성합니다.

### 선택 주제 (하나 선택)

- **A. 음성 수업 질문 도우미**: '샘플링이 뭐예요?'라고 질문하면 간단히 설명
- **B. 교내 안내 도우미**: '수업 준비물을 알려줘' 등의 질문에 미리 정한 안내 내용을 바탕으로 답변하도록 지시
- **C. 업무 음성 메모 도우미**: 짧게 녹음한 메모를 텍스트로 바꾸고 핵심만 요약한 후 읽어 주기

### 진행 순서

1. 자신의 주제를 한 문장으로 적습니다.
2. 한국어 음성 파일 1개를 준비합니다(본인 음성 권장, 어려우면 합성 음성 사용).
3. API 호출 전에 음성 파일 존재 여부와 API 키 설정 여부를 확인합니다.
4. `run_voice_assistant()`를 실행합니다. C 주제는 LLM 지시문을 '요약'으로 수정합니다.
5. 인식 문장·답변 문장을 화면에 보여주고 음성 파일을 재생합니다.
6. 기대와 다른 결과가 있다면 어느 단계(STT/LLM/TTS)에서 차이가 생겼는지 기록합니다.

### 제출물
- 실행 노트북 (`.ipynb`, API 키 절대 포함 금지)
- 입력 WAV 1개와 출력 WAV 1개 (**공유 동의가 있는 경우에만 제출**)
- 결과 요약 3줄: '입력 음성 내용 / 인식 결과 / 답변 결과'
- 개선할 점 1개: 정확도, 발화 속도, 요금, 예외 처리 등

> **추가 도전:** STT 결과와 LLM 결과를 모두 JSON 파일로 저장합니다. 가능하다면 오류 파일을 입력했을 때 사용자에게 안내 메시지가 출력되도록 구현해 보세요.

In [ ]:
# 실제 음성 AI 처리 결과를 JSON 파일로 기록합니다.
record = {
    'input_file': str(FINAL_INPUT),
    'transcribed_text': result['question'],
    'assistant_answer': result['answer'],
    'output_file': str(result['audio_file']),
}
report_path = REPORT_DIR / 'voice_assistant_record.json'
report_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding='utf-8')
print('기록 저장:', report_path)

# 마무리: 이제 음성 모델의 내부 원리를 공부할 차례입니다

| 이번 API 실습에서 경험한 것 | 다음 음성 모델 수업에서 답할 질문 |
|---|---|
| WAV 파일을 서버에 업로드 | 음성은 어떻게 숫자로 저장되는가? 샘플링·채널이란? |
| Whisper가 문장을 인식 | 음성에서 어떤 특징을 추출할까? FFT, STFT, Mel은 왜 필요할까? |
| 음성마다 인식 결과가 달라짐 | 데이터·정답·오류율(CER/WER)을 어떻게 다룰까? |
| 문장을 음성으로 생성 | TTS는 어떤 방식으로 음성을 합성할까? |
| API를 이용해 여러 기능 연결 | 모델의 입력·출력 구조와 실제 시스템은 어떻게 연결될까? |

### 반드시 구분하세요

- **OpenAI API 사용:** 완성된 외부 모델 서비스를 호출하는 것
- **Whisper 내부 학습·추론 원리:** 음성 특징과 모델 구조를 이해하는 것
- **모델을 직접 학습시키기:** 학습 데이터, 손실 함수, 최적화, 평가까지 수행하는 것

이번 API 실습에서는 첫 번째를 경험합니다. 이어지는 음성 모델 수업에서는 두 번째를 중심으로 공부하고, 필요에 따라 세 번째도 살펴봅니다.

### 학습 점검 체크리스트

- [ ] API와 모델의 차이를 설명할 수 있다.
- [ ] API 키를 공개하지 않고 설정할 수 있다.
- [ ] 한국어 텍스트를 음성 WAV로 합성할 수 있다.
- [ ] Whisper로 한국어 WAV를 받아쓰기할 수 있다.
- [ ] STT → LLM → TTS 전체 흐름을 설명할 수 있다.
- [ ] 인식 결과를 실제 발화 정답과 비교하고 차이를 기록할 수 있다.

### 참고 문서 (사용 모델·요금·지원 매개변수는 실습 당일 확인)
- [OpenAI API 시작하기](https://platform.openai.com/docs/quickstart)
- [Speech to text](https://platform.openai.com/docs/guides/speech-to-text)
- [Text to speech](https://platform.openai.com/docs/guides/text-to-speech)
- [API 가격](https://openai.com/api/pricing/)